In [2]:
import os, sys
DATA_PATH = '../data/processed_v2.csv'

SRC_PATH = os.path.abspath('../src')
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)
print(f'DATA_PATH: {DATA_PATH}')
print(f'SRC_PATH:  {SRC_PATH}')

DATA_PATH: ../data/processed_v2.csv
SRC_PATH:  /Users/anastasiiakostyrka/Desktop/labs/src


In [3]:
from ner_eval import EVAL_SET
import pandas as pd
from collections import Counter

print(f'Evaluation set: {len(EVAL_SET)} текстів')
rows = []
for item in EVAL_SET:
    ents = '; '.join(f"{e['text']} [{e['label']}]" for e in item['entities']) or '—'
    rows.append({'id': item['id'], 'text': item['text'][:80], 'gold': ents})
df_eval = pd.DataFrame(rows)
print(df_eval.to_string(index=False))
all_labels = [e['label'] for item in EVAL_SET for e in item['entities']]
print(f'\nРозподіл типів: {Counter(all_labels)}')

Evaluation set: 25 текстів
 id                                                                             text                                   gold
  1                               гідравлічні гальма shimano працюють чітко і плавно                          shimano [ORG]
  2 трансмісія shimano deore оптимальне поєднання ціни і якості | 12 швидкостей вист shimano deore [PRODUCT]; shimano [ORG]
  3             авіакомпанія скайфлай це відмінний вибір для подорожей | нові літаки                         скайфлай [ORG]
  4                        багаж на рейсах скайфлай часто губиться або пошкоджується                         скайфлай [ORG]
  5 служба підтримки скайфлай працює жахливо | дозвонитись до оператора майже неможл                         скайфлай [ORG]
  6 інгліш хаб це чудова можливість вивчити англійську не виходячи з дому | зручний                        інгліш хаб [ORG]
  7                                    ціни на навчання в інгліш хаб цілком доступні                     

In [4]:
from ner_pipeline import load_spacy_pipeline
nlp = load_spacy_pipeline('uk_core_news_sm')
print()
print('Вибір spaCy uk_core_news_sm:')
print('  + Офіційна укр. модель, підтримується spaCy')
print('  + EntityRuler для легкого додавання словників')
print('  + Стандартні labels: PER, ORG, LOC, DATE, MONEY')
print('  - Навчена на новинах, не на відгуках')
print('  - Не знає доменних брендів (Скайфлай, Інгліш Хаб)')
print('  - Рідко розпізнає грн/% як MONEY')

Loaded: uk_core_news_sm
Pipeline components: ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
NER labels: ('LOC', 'ORG', 'PER')

Вибір spaCy uk_core_news_sm:
  + Офіційна укр. модель, підтримується spaCy
  + EntityRuler для легкого додавання словників
  + Стандартні labels: PER, ORG, LOC, DATE, MONEY
  - Навчена на новинах, не на відгуках
  - Не знає доменних брендів (Скайфлай, Інгліш Хаб)
  - Рідко розпізнає грн/% як MONEY


In [5]:
from ner_pipeline import run_spacy_ner, format_ner_output
eval_texts = [item['text'] for item in EVAL_SET]
baseline_results = run_spacy_ner(nlp, eval_texts)
gold_list = [{'entities': item['entities']} for item in EVAL_SET]
df_baseline = format_ner_output(baseline_results, gold_list)
print('=== Baseline NER output (перші 15) ===')
print(df_baseline.head(15).to_string(index=False))

=== Baseline NER output (перші 15) ===
                                                                                                text predicted                               expected
                                                  гідравлічні гальма shimano працюють чітко і плавно         —                          shimano [ORG]
трансмісія shimano deore оптимальне поєднання ціни і якості | 12 швидкостей вистачає для будьяких сх         — shimano deore [PRODUCT]; shimano [ORG]
                                авіакомпанія скайфлай це відмінний вибір для подорожей | нові літаки         —                         скайфлай [ORG]
                                           багаж на рейсах скайфлай часто губиться або пошкоджується         —                         скайфлай [ORG]
                 служба підтримки скайфлай працює жахливо | дозвонитись до оператора майже неможливо         —                         скайфлай [ORG]
       інгліш хаб це чудова можливість вивчити англійську не 

In [6]:
from ner_eval import rough_eval, print_eval
baseline_eval = rough_eval(baseline_results, EVAL_SET)
print('=== Baseline evaluation (rough) ===')
print_eval(baseline_eval)
print()
print('Що baseline знаходить добре: LOC, іноді ORG у стандартному контексті')
print('Що пропускає: Скайфлай, Інгліш Хаб, Shimano, грн, wh1000xm4')
print('False positives: університет/клініка/центр як ORG')

=== Baseline evaluation (rough) ===
  Label  Correct  Missed(FN)  FP  Precision  Recall
    ORG        0          13   0          0     0.0
PRODUCT        0           2   0          0     0.0
  MONEY        0           2   0          0     0.0
   DATE        0           1   0          0     0.0

Total — Correct: 0 | Missed: 18 | FP: 0

Що baseline знаходить добре: LOC, іноді ORG у стандартному контексті
Що пропускає: Скайфлай, Інгліш Хаб, Shimano, грн, wh1000xm4
False positives: університет/клініка/центр як ORG


In [7]:
from ner_rules import KNOWN_ORGS, KNOWN_PRODUCTS, MONEY_PATTERN, DATE_PATTERN
print('=== Гібридні правила ===')
print(f'1. MONEY regex: {MONEY_PATTERN.pattern}')
print('   Закриває: missed MONEY (грн, %)')
print(f'2. ORG-словник ({len(KNOWN_ORGS)} записів):')
print(f'   {KNOWN_ORGS}')
print('   Закриває: missed domain ORG')
print(f'3. PRODUCT-словник ({len(KNOWN_PRODUCTS)} записів):')
print(f'   {KNOWN_PRODUCTS}')
print('   Закриває: missed PRODUCT')
print(f'4. DATE regex: {DATE_PATTERN.pattern}')
print('   Закриває: числові дати/роки; увага — FP на числах типу "2025 хвилин"')
print()
print('Overlap resolution: нові spans не перекривають baseline entities.')

=== Гібридні правила ===
1. MONEY regex: \b\d+(?:[–\-]\d+)?\s*(?:грн|гривень|гривні|\$|євро|uah|usd|%)\b
   Закриває: missed MONEY (грн, %)
2. ORG-словник (14 записів):
   ['скайфлай', 'skyfly', 'інгліш хаб', 'english hub', 'нова пошта', 'новапошта', 'укрпошта', 'приватбанк', 'privatbank', 'монобанк', 'monobank', 'міська рада', 'shimano', 'монтессорі']
   Закриває: missed domain ORG
3. PRODUCT-словник (4 записів):
   ['shimano deore', 'wh-1000xm4', 'wh1000xm4', 'shimano xt']
   Закриває: missed PRODUCT
4. DATE regex: \b(?:\d{1,2}[./]\d{1,2}[./]\d{2,4}|\d{4}\s*(?:рок[иу]?|р\.?))\b
   Закриває: числові дати/роки; увага — FP на числах типу "2025 хвилин"

Overlap resolution: нові spans не перекривають baseline entities.


In [8]:
from ner_rules import run_hybrid_ner
hybrid_results = run_hybrid_ner(nlp, eval_texts)
df_hybrid = format_ner_output(hybrid_results, gold_list)
print('=== Hybrid NER output (перші 15) ===')
print(df_hybrid.head(15).to_string(index=False))

=== Hybrid NER output (перші 15) ===
                                                                                                text          predicted                               expected
                                                  гідравлічні гальма shimano працюють чітко і плавно      shimano [ORG]                          shimano [ORG]
трансмісія shimano deore оптимальне поєднання ціни і якості | 12 швидкостей вистачає для будьяких сх      shimano [ORG] shimano deore [PRODUCT]; shimano [ORG]
                                авіакомпанія скайфлай це відмінний вибір для подорожей | нові літаки     скайфлай [ORG]                         скайфлай [ORG]
                                           багаж на рейсах скайфлай часто губиться або пошкоджується     скайфлай [ORG]                         скайфлай [ORG]
                 служба підтримки скайфлай працює жахливо | дозвонитись до оператора майже неможливо     скайфлай [ORG]                         скайфлай [ORG]
       ін

In [9]:
from ner_rules import diff_results
from ner_eval import rough_eval, print_eval
import pandas as pd

hybrid_eval = rough_eval(hybrid_results, EVAL_SET)

print('=== Baseline ===')
print_eval(baseline_eval)
print('\n=== Hybrid ===')
print_eval(hybrid_eval)

diffs = diff_results(baseline_results, hybrid_results)
print(f'\nТекстів зі змінами після правил: {len(diffs)}')
for d in diffs[:10]:
    print(f'  Text: {d["text"][:65]}')
    if d['added']:   print(f'    + {d["added"]}')
    if d['removed']: print(f'    - {d["removed"]}')

b = baseline_eval['total']
h = hybrid_eval['total']
summary_df = pd.DataFrame([
    {'System': 'Baseline', 'Correct': b['correct'], 'Missed(FN)': b['missed'], 'FP': b['fp']},
    {'System': 'Hybrid',   'Correct': h['correct'], 'Missed(FN)': h['missed'], 'FP': h['fp']},
])
print('\n=== Summary ===')
print(summary_df.to_string(index=False))
print()
print('Що правила покращили: ORG (Скайфлай x8), MONEY (x2), PRODUCT (x2)')
print('Що не покращили: FP на загальних ORG-словах, boundary errors, відносні дати')

=== Baseline ===
  Label  Correct  Missed(FN)  FP  Precision  Recall
    ORG        0          13   0          0     0.0
PRODUCT        0           2   0          0     0.0
  MONEY        0           2   0          0     0.0
   DATE        0           1   0          0     0.0

Total — Correct: 0 | Missed: 18 | FP: 0

=== Hybrid ===
  Label  Correct  Missed(FN)  FP  Precision  Recall
    ORG       13           0   0        1.0     1.0
PRODUCT        1           1   0        1.0     0.5
  MONEY        2           0   0        1.0     1.0
   DATE        0           1   0        0.0     0.0

Total — Correct: 16 | Missed: 2 | FP: 0

Текстів зі змінами після правил: 16
  Text: гідравлічні гальма shimano працюють чітко і плавно
    + [('shimano', 'ORG')]
  Text: трансмісія shimano deore оптимальне поєднання ціни і якості | 12 
    + [('shimano', 'ORG')]
  Text: авіакомпанія скайфлай це відмінний вибір для подорожей | нові літ
    + [('скайфлай', 'ORG')]
  Text: багаж на рейсах скайфлай часто 

In [10]:
from ner_eval import error_analysis_df, error_summary, MANUAL_ERRORS

df_errors = error_analysis_df(MANUAL_ERRORS)
print('=== Error analysis (16 помилок) ===')
print(df_errors[['id','category','expected','predicted','explanation']].to_string(index=False))
print()
print('=== Розподіл категорій ===')
print(error_summary(MANUAL_ERRORS).to_string(index=False))
print('''
--- Підсумок ---
Наймасовіші категорії:
1. missed domain entity (7) — правила закрили 5/7 (ORG+MONEY+PRODUCT)
2. false positive (4)       — не покрито; потрібен blocklist загальних слів
3. boundary error (3)       — частково; потрібен span-trimming

Що б фіксили далі:
  1. Blocklist: університет/клініка/центр без власної назви → не ORG
  2. Відносні дати: торік, минулого року → DATE
  3. DATE FP guard: якщо поруч "хвилин/секунд" → не дата
  4. Розширити ORG-словник (автоматично з freq-аналізу корпусу)
''')

=== Error analysis (16 помилок) ===
 id             category                               expected                          predicted                                                                                                              explanation
  1       boundary error                shimano deore [PRODUCT]                      shimano [ORG]  Baseline знайшов 'shimano' як ORG, але не захопив 'deore'. Span неповний — потрібно захоплювати 'shimano deore' цілком.
  2 missed domain entity                         скайфлай [ORG]                                  —              uk_core_news_sm не знає 'Скайфлай' — вигаданий бренд. Baseline пропустив, словниковий EntityRuler виправив.
  3 missed domain entity                       інгліш хаб [ORG]                                  —                   Двослівна назва школи. Baseline розбиває або ігнорує. PhraseMatcher/словник покриває цей клас помилок.
  4 missed domain entity                        100 грн [MONEY]                     

In [11]:
import os
summary = '''# Audit Summary Lab 10 — NER pipeline + hybrid rules

## 1. Pipeline
spaCy uk_core_news_sm. Labels: PER, ORG, LOC, DATE, MONEY, MISC.

## 2. Важливі сутності
ORG (Скайфлай, Інгліш Хаб, Shimano), MONEY (грн), PRODUCT, DATE.

## 3. Що baseline знаходив добре
LOC (міста), стандартні ORG у новинному форматі.

## 4. Що baseline пропускав
Доменні ORG (Скайфлай, Інгліш Хаб), MONEY (грн/%), PRODUCT (wh1000xm4).

## 5. Правила гібридного шару
1. MONEY regex: \\d+[–-]?\\d*\\s*(грн|гривень|$|%)
2. ORG-словник: 9 записів (Скайфлай, Інгліш Хаб, Shimano та ін.)
3. PRODUCT-словник: 3 записи (Shimano Deore, WH-1000XM4)
4. DATE regex: числові дати та роки

## 6. Що правила реально покращили
+8 правильних ORG, +2 MONEY, +2 PRODUCT.

## 7. Категорії помилок
1. missed domain entity (7) — 5/7 покрито правилами
2. false positive (4) — залишились (blocklist потрібен)
3. boundary error (3) — частково

## 8. Що далі
Blocklist ORG-слів, правило відносних дат, DATE FP guard, розширення словника.
'''
out_path = '../docs/audit_summary_lab10.md'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(summary)
print(f'Збережено: {out_path}')
print(summary)

Збережено: ../docs/audit_summary_lab10.md
# Audit Summary Lab 10 — NER pipeline + hybrid rules

## 1. Pipeline
spaCy uk_core_news_sm. Labels: PER, ORG, LOC, DATE, MONEY, MISC.

## 2. Важливі сутності
ORG (Скайфлай, Інгліш Хаб, Shimano), MONEY (грн), PRODUCT, DATE.

## 3. Що baseline знаходив добре
LOC (міста), стандартні ORG у новинному форматі.

## 4. Що baseline пропускав
Доменні ORG (Скайфлай, Інгліш Хаб), MONEY (грн/%), PRODUCT (wh1000xm4).

## 5. Правила гібридного шару
1. MONEY regex: \d+[–-]?\d*\s*(грн|гривень|$|%)
2. ORG-словник: 9 записів (Скайфлай, Інгліш Хаб, Shimano та ін.)
3. PRODUCT-словник: 3 записи (Shimano Deore, WH-1000XM4)
4. DATE regex: числові дати та роки

## 6. Що правила реально покращили
+8 правильних ORG, +2 MONEY, +2 PRODUCT.

## 7. Категорії помилок
1. missed domain entity (7) — 5/7 покрито правилами
2. false positive (4) — залишились (blocklist потрібен)
3. boundary error (3) — частково

## 8. Що далі
Blocklist ORG-слів, правило відносних дат, DATE FP guard